# Droplet rebound simulation - Postprocessing

This notebook is used to postprocess the simulations for the droplet rebound test case.

In [1]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
// BoSSSshell.WorkflowMgm.Init("DropletRebound");
// var sessions = BoSSSshell.WorkflowMgm.Sessions;

In [ ]:
//OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletRebound");
//OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound");
OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier");

Opening existing database '\\dc3\userspace\smuda\Databases\XNSEpaper_RisingBubble'.


In [3]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80*	3/23/2020 10:05:28 PM	931391e0...
#1: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20	3/23/2020 10:05:20 PM	d3c1cfe5...
#2: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40	3/23/2020 10:05:22 PM	44504bf6...
#3: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart1*	4/17/2020 2:43:50 PM	7a7385d1...
#4: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart2*	5/31/2020 8:10:24 PM	97a64a0d...
#5: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart3*	6/1/2020 4:00:14 PM	42b883f6...
#6: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart4*	6/3/2020 10:12:02 PM	22177ad6...
#7: RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart5	6/5/2020 9:21:58 PM	6e3cf7d8...
#8: RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh20*	5/8/2020 6:45:50 PM	8351c0ba...
#9: RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh20_restart1	5/9/2020 8:49:47 AM	4c450744...
#10: RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh10_ReInit50	6/5/202

In [ ]:
var select = sessions.Where(sess => sess.Name.Contains("DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart2"));
select

#0: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart2*	04/16/2024 11:43:08	005048e6...


In [4]:
// var sess = select.Pick(0);
var sess = sessions.Pick(16);
sess

RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh60	3/26/2020 9:43:00 AM	f6cdd826...

In [ ]:
//System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "1");

In [ ]:
//sess.Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble__RisingBubble_tc1_ConvStudy_k3_mesh60__f6cdd826-b8b5-47b0-8e21-ae9a79897471


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble__RisingBubble_tc1_ConvStudy_k3_mesh60__f6cdd826-b8b5-47b0-8e21-ae9a79897471

In [ ]:
List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
restartSessionList.Add(sess);
Guid restartID = sess.RestartedFrom;
while(restartID != Guid.Empty) {
    var restartSession = sessions.Where(sess => sess.ID == restartID).Single();
    restartSessionList.Add(restartSession);
    restartID = restartSession.RestartedFrom;
}
restartSessionList.Reverse();
restartSessionList

#0: DropletRebound_Gauthier	DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5*	02/18/2025 16:49:12	ccada4b5...
#1: DropletRebound_Gauthier	DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5_LineSearch_restart1*	02/20/2025 10:01:02	bb733c4c...


In [ ]:
//restartSessionList.Pick(2).Timesteps

In [ ]:
List<ITimestepInfo> timestepList = new List<ITimestepInfo>();
foreach (var sess in restartSessionList) {
    int firstIndex = sess.Timesteps.Pick(0).TimeStepNumber.MajorNumber;
    int foundIndex = timestepList.FindIndex(ts => ts.TimeStepNumber.MajorNumber == firstIndex);
    if (foundIndex > -1) {
        timestepList.RemoveRange(foundIndex, timestepList.Count - foundIndex);
    }
    timestepList.AddRange(sess.Timesteps);
} 
//timestepList

In [ ]:
timestepList.Count

326

In [ ]:
timestepList.Take(10)

#0:  { Time-step: 0.0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, MPIrank, CutCells; Name:  }
#1:  { Time-step: 0.1; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, MPIrank, CutCells; Name:  }
#2:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }
#3:  { Time-step: 1; Physical time: 2E-05s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }
#4:  { Time-step: 2; Physical time: 4E-05s; Fields: Phi, PhiDG, VelocityX, VelocityY

In [ ]:
//sess.KeysAndQueries["Option_LevelSetEvolution"]
//sess.GetSessionDirectory()
//sess.RestartedFrom

\\hpccluster\hpccluster-scratch\smuda\DropletRebound\sessions\d2c5673f-901d-42d7-8d6a-856bd9e8865d

In [ ]:
//sess.Timesteps.Skip(2).Every(10).Export().WithSupersampling(2).Do()
//sess.Timesteps.Every(2).Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound__DropletRebound_8x8x8AMR1_dropVelocity100vH_restart2_ReInit__b52120d3-bd87-49f5-851f-fbd1c7ebe548


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound__DropletRebound_8x8x8AMR1_dropVelocity100vH_restart2_ReInit__b52120d3-bd87-49f5-851f-fbd1c7ebe548

## Droplet metrics (center of mass, volume, sphericity) over time

In [ ]:
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;

In [ ]:
var sessTimesteps = timestepList.Skip(2).Every(5);
sessTimesteps.Take(5)

#0:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }
#1:  { Time-step: 5; Physical time: 0.0001s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }
#2:  { Time-step: 10; Physical time: 0.0002s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }
#3:  { Time-step: 15; Physical time: 0.00030000000000000003s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

In [ ]:
int timesteps = sessTimesteps.Count();
timesteps

65

In [ ]:
int D = 2;

In [ ]:
double[] time = new double[timesteps];
MultidimensionalArray centerOfMass = MultidimensionalArray.Create(timesteps, D);
double[] volume = new double[timesteps];
double[] sphericity = new double[timesteps];
double[] minDropPos = new double[timesteps];

In [ ]:
int degPhi = sessTimesteps.ElementAt(0).GetField("Phi").Basis.Degree;
int quadOrder = (2 * degPhi) * 2 + 1;   // using Saye

In [ ]:
sessTimesteps.ElementAt(0).GetField("Phi").Basis.GridDat.iGeomCells.Count

725

In [ ]:
for (int ts = 0; ts < timesteps; ts++) {
    var tStep = sessTimesteps.ElementAt(ts);

    time[ts] = tStep.PhysicalTime;

    SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
    GridData grdData = (GridData)phi.GridDat; 
    LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
    levelSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
    LsTrk.UpdateTracker(tStep.PhysicalTime);

    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), quadOrder, 1).XQuadSchemeHelper;
    SpeciesId spcId = LsTrk.GetSpeciesId("A");
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcId);


    // volume
    double dropVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
            dropVolume += ResultsOfIntegration[i, 0];
        }
    ).Execute();

    volume[ts] = dropVolume;


    // center of mass
    double[] dropCoM = new double[D];
    CellQuadrature.GetQuadrature(new int[] { D }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            NodeSet nodes_global = QR.Nodes.CloneAs();
            for (int i = i0; i < i0 + Length; i++) {
                LsTrk.GridDat.TransformLocal2Global(QR.Nodes, nodes_global, i);
                EvalResult.AccSubArray(1.0, nodes_global, new int[] { i - i0, -1, -1 });
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int d = 0; d < D; d++) {
                for(int i = 0; i < Length; i++)
                    dropCoM[d] += ResultsOfIntegration[i, d];
            }
        }
    ).Execute();

    for(int d = 0; d < D; d++) {
        centerOfMass[ts,d] = dropCoM[d] / dropVolume;
    }

    // minimal drop position
    double minZCoordDrop = 1.0;
    MultidimensionalArray InterfacePoints = XNSEUtils.GetInterfacePoints(LsTrk, levelSet);
    for(int i = 0; i < InterfacePoints.Lengths[0]; i++) {
        double zCoord = InterfacePoints[i, D-1];
        if (zCoord < minZCoordDrop)
            minZCoordDrop = zCoord;
    }
    minDropPos[ts] = minZCoordDrop;

    // // sphericity
    // double dropSurface = 0.0;
    // CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask());
    // CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    //     cqs.Compile(LsTrk.GridDat, quadOrder),
    //     delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
    //         EvalResult.SetAll(1.0);
    //     },
    //     delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
    //         for (int i = 0; i < Length; i++)
    //         dropSurface += ResultsOfIntegration[i, 0];
    //     }
    // ).Execute();

    // sphericity[ts] = Math.Pow(Math.PI, 1.0/3.0) * Math.Pow(6*dropVolume, 2.0/3.0) / dropSurface;

}

In [ ]:
centerOfMass

Dimension,Lengths,Storage,IsContinuous,StructureType,Length,NoOfCols,NoOfRows,IsLocked
2,"[ 29, 3 ]","[ 0.0779999999998405, 2.558302989516735E-14, 0.0006000009202340767, 0.07800000913097228, 1.141143452576779E-08, 0.00058877673692869, 0.07800004582745412, 6.165827891578793E-08, 0.0005775597410619704, 0.0780001068095097, 1.4472221693822214E-07, 0.0005663245828036336, 0.07800017816814982, 2.4068444588071557E-07, 0.0005550123300021596, 0.07800025632900907, 3.471196188637224E-07, 0.0005435805512960328, 0.07800037765669761, 5.178751200431484E-07, 0.0005318338165095191, 0.07800053130659605, 7.484683299922748E-07, 0.0005195933496623941, 0.07800058423527083, 8.374319550857631E-07, 0.0005078902749297473, 0.07800069705703166, 1.09371803060473E-06, 0.0004959659491326437, 0.07800079562791382, 1.4111064471417318E-06, 0.00048422340447693067, 0.07800090965978712, 1.7843403922798256E-06, 0.00047230633802986693, 0.07800104657871107, 2.1802657803354298E-06, 0.0004603550717183633, 0.07800119023167944, 2.602109620116846E-06, 0.00044839098124407575, 0.07800134377474834, 3.0654891316834105E-06, 0.0004363608212228924, 0.07800158224409139, 3.4252175995495027E-06, 0.00042436068997999657, 0.07800204950532191, 3.7130648090299577E-06, 0.0004125044613356929, 0.07800273355766432, 4.039330465528441E-06, 0.0004008267485342021, 0.07800366983860615, 4.537554218702028E-06, 0.0003894459529264859, 0.07800525085960763, 5.240186913454737E-06, 0.00037870948902141453, 0.07800573312982935, 5.61199870197617E-06, 0.0003677452049509152, 0.07800720861689761, 7.158742380570249E-06, 0.000358033031406997, 0.07800813141674631, 8.929575783688386E-06, 0.000344502787288337, 0.07800966818416495, 1.1550810168460266E-05, 0.0003298664995958589, 0.07801139299269286, 1.5061336944306873E-05, 0.0003156716255310446, 0.07801217460644314, 1.790644864682405E-05, 0.00030817251857992456, 0.0780128005392319, 2.023097874636097E-05, 0.00030049766689954085, 0.07801342888964685, 2.1920199429047637E-05, 0.0002924148157341421, 0.07801406712707451, 2.3229394408869595E-05, 0.0002842664003510255 ]",True,General,87,3,29,False


In [ ]:
var fmt = new PlotFormat();    
fmt.LineColor = LineColors.Blue;

var gp = new Gnuplot();
gp.SetMultiplot(1,2);

Plot2Ddata plt = new Plot2Ddata(); 
plt.AddDataGroup("centerOfMassZ", time, centerOfMass.ExtractSubArrayShallow(-1, D-1).To1DArray(), fmt);
gp.SetSubPlot(0, 0, "centerOfMassZ");
plt.ToGnuplot(gp);

// plt = new Plot2Ddata(); 
// plt.AddDataGroup("minimal Z position", time, minDropPos, fmt);
// gp.SetSubPlot(0, 1, "minZposDrop");
// plt.ToGnuplot(gp);

// plt = new Plot2Ddata(); 
// plt.AddDataGroup("sphericity", time, sphericity, fmt);
// gp.SetSubPlot(0, 1, "sphericity");
// plt.ToGnuplot(gp);

//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:500)

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.00105 
 
 
 
 
 0.0011 
 
 
 
 
 0.00115 
 
 
 
 
 0.0012 
 
 
 
 
 0.00125 
 
 
 
 
 0.0013 
 
 
 
 
 0.00135 
 
 
 
 
 0.0014 
 
 
 
 
 0.00145 
 
 
 
 
 0.0015 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 centerOfMassZ 
 
 
 centerOfMassZ 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M475.5,62.1 L528.9,62.1 M94.5,41.1 L103.0,58.2 L111.6,71.1 L120.1,88.3 L128.7,105.7 L137.2,123.1
 L145.8,140.5 L154.3,157.9 L162.9,175.5 L171.4,193.0 L179.9,210.5 L188.5,228.1 L197.0,245.6 L205.6,263.1
 L214.1,280.5 L222.7,297.8 L231.2,315.0 L239.8,331.7 L248.3,348.1 L256.8,363.8 L265.4,378.8 L273.9,389.4
 L282.5,402.2 L291.0,413.5 L299.6,423.5 L308.1,429.6 L316.7,435.8 L325.2,440.3 L333.7,442.8 L342.3,443.6
 L350.8,442.8 L359.4,440.8 L367.9,437.7 L376.5,433.4 L385.0,428.5 L393.6,422.9 L402.1,418.5 L410.6,412.5
 L419.2,406.3 L427.7,399.9 L436.3,393.3 L444.8,386.5 L453.4,379.6 L461.9,372.6 L470.4,367.4 L479.0,360.3
 L487.5,353.2 L496.1,346.1 L504.6,339.2 L513.2,334.1 L521.7,327.2 L530.3,320.6 L538.8,314.0 L547.3,307.6
 L555.9,301.3 L564.4,295.2 L573.0,289.1 L581.5,283.3 L590.1,277.5 L598.6,271.9 L607.2,266.5 L615.7,261.2
 L624.2,256.1 L632.8,251.1 L641.3,246.2 '/>

In [ ]:
sess.Name

DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4

In [ ]:
plt.SaveToTextFile("results/" + sess.Name + "_CenterOfMassZ")

## mean vertical velocity / lift over time

In [ ]:
double[] time = new double[timesteps];
double[] meanVertVelocity = new double[timesteps];
// double[] lift = new double[timesteps];
MultidimensionalArray pressureForce = MultidimensionalArray.Create(timesteps, D);

In [ ]:
int degPhi = sess.Timesteps.ElementAt(0).GetField("Phi").Basis.Degree;
int quadOrder = (2 * degPhi) * 2 + 1;   // using Saye

In [ ]:
for (int ts = 0; ts < timesteps; ts++) {
    var tStep = sessTimesteps.ElementAt(ts);

    time[ts] = tStep.PhysicalTime;

    SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
    GridData grdData = (GridData)phi.GridDat; 
    LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
    levelSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
    LsTrk.UpdateTracker(tStep.PhysicalTime);

    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), quadOrder, 1).XQuadSchemeHelper;
    SpeciesId spcId = LsTrk.GetSpeciesId("A");
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcId);


    // volume
    double dropVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
                dropVolume += ResultsOfIntegration[i, 0];
        }
    ).Execute();


    // mean vertical velocity
    double vertVel = 0.0;
    DGField velocityVertical = ((XDGField)tStep.GetField("VelocityY")).GetSpeciesShadowField("A");
    if (D == 3)
        velocityVertical = ((XDGField)tStep.GetField("VelocityZ")).GetSpeciesShadowField("A");

    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            velocityVertical.Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1,-1,0), 1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
                vertVel += ResultsOfIntegration[i, 0];
        }
    ).Execute();

    meanVertVelocity[ts] = vertVel / dropVolume;


    // lift
    double[] pForce = new double[D];
    DGField pressure = ((XDGField)tStep.GetField("Pressure")).GetSpeciesShadowField("B");
    CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask());
    CellQuadrature.GetQuadrature(new int[] { D }, LsTrk.GridDat,
        cqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            var normals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, Length);
            int K = EvalResult.GetLength(1);
            var pressureEval = MultidimensionalArray.Create(Length, K);
            pressure.Evaluate(i0, Length, QR.Nodes, pressureEval, 1.0);  
            for(int d = 0; d < D; d++) {              
                for (int j = 0; j < Length; j++) {
                    for (int k = 0; k < K; k++) {
                        EvalResult[j, k, d] = pressureEval[j, k] * normals[j, k, d];
                    }
                }
            }
                
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int d = 0; d < D; d++) {  
                for (int i = 0; i < Length; i++)
                    pForce[d] += ResultsOfIntegration[i, d];
            }
        }
    ).Execute();

    for(int d = 0; d < D; d++)
        pressureForce[ts, d] = pForce[d];

}

In [ ]:
var fmt = new PlotFormat();    
fmt.LineColor = LineColors.Blue;

var gp = new Gnuplot();
gp.SetMultiplot(1,2);

Plot2Ddata plt = new Plot2Ddata(); 
plt.AddDataGroup("meanVelocityZ", time, meanVertVelocity, fmt);
gp.SetSubPlot(0, 0, "meanVelocityZ");
plt.ToGnuplot(gp);

plt = new Plot2Ddata(); 
plt.AddDataGroup("lift", time, pressureForce.ExtractSubArrayShallow(-1, D-1).To1DArray(), fmt);
gp.SetSubPlot(0, 1, "lift");
plt.ToGnuplot(gp);

//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:500)

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.25 
 
 
 
 
 -0.2 
 
 
 
 
 -0.15 
 
 
 
 
 -0.1 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 meanVelocityZ 
 
 
 meanVelocityZ 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M475.5,62.1 L528.9,62.1 M69.6,435.2 L78.5,436.6 L87.4,438.3 L96.3,439.5 L105.2,440.5 L114.1,441.3
 L123.0,442.2 L131.9,442.8 L140.8,443.3 L149.7,443.6 L158.6,443.7 L167.5,443.5 L176.4,442.9 L185.3,441.9
 L194.2,441.0 L203.1,439.2 L212.0,435.0 L220.9,428.3 L229.8,419.5 L238.7,408.3 L247.6,391.4 L256.5,369.5
 L265.4,344.5 L274.3,320.5 L283.2,291.3 L292.1,261.4 L301.0,233.1 L309.9,202.2 L318.8,173.7 L327.7,146.5
 L336.6,125.5 L345.5,109.2 L354.4,93.2 L363.3,81.4 L372.2,69.7 L381.1,62.3 L390.0,59.3 L398.9,57.5
 L407.8,55.0 L416.7,52.3 L425.6,49.7 L434.5,47.5 L443.4,46.0 L452.3,44.8 L461.2,44.1 L470.1,44.1
 L479.0,44.6 L487.9,45.6 L496.8,47.0 L505.7,48.7 L514.6,50.5 L523.5,52.6 L532.4,54.8 L541.3,57.0
 L550.2,59.2 L559.1,61.4 L568.0,63.6 L576.9,65.8 L585.8,68.1 L594.7,70.5 L603.6,72.8 L612.5,75.1
 L621.4,77.4 L630.3,79.7 L639.2,82.0 '/> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.7 
 
 
 
 
 -0.6 
 
 
 
 
 -0.5 
 
 
 
 
 -0.4 
 
 
 
 
 -0.3 
 
 
 
 
 -0.2 
 
 
 
 
 -0.1 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 lift 
 
 
 lift 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M1325.4,62.1 L1378.8,62.1 M811.3,93.4 L820.3,86.8 L829.3,90.0 L838.4,94.2 L847.4,95.8 L856.4,96.1
 L865.4,97.5 L874.4,98.8 L883.4,100.5 L892.5,102.6 L901.5,105.2 L910.5,110.1 L919.5,114.4 L928.5,115.8
 L937.6,115.1 L946.6,128.0 L955.6,154.6 L964.6,185.0 L973.6,187.9 L982.7,239.5 L991.7,286.4 L1000.7,300.0
 L1009.7,313.4 L1018.7,322.8 L1027.7,399.6 L1036.8,394.1 L1045.8,406.9 L1054.8,447.8 L1063.8,395.9 L1072.8,352.1
 L1081.9,286.6 L1090.9,301.0 L1099.9,258.1 L1108.9,214.2 L1117.9,204.8 L1127.0,157.1 L1136.0,120.5 L1145.0,128.6
 L1154.0,139.4 L1163.0,135.3 L1172.0,134.3 L1181.1,128.6 L1190.1,123.2 L1199.1,117.6 L1208.1,110.0 L1217.1,102.9
 L1226.2,97.9 L1235.2,93.3 L1244.2,89.4 L1253.2,87.6 L1262.2,85.7 L1271.2,83.9 L1280.3,82.3 L1289.3,82.3
 L1298.3,82.2 L1307.3,82.1 L1316.3,82.1 L1325.4,82.1 L1334.4,81.0 L1343.4,81.4 L1352.4,80.8 L1361.4,81.2
 L1370.5,81.6 L1379.5,81.2 L1388.5,80.6 '/>

In [ ]:
plt.SaveToTextFile("results/" + sess.Name + "_meanVelocityZ")